In [1]:
import yaml
import os
import matplotlib.pyplot as plt 
import seaborn as sns
import os
import yaml
import os.path as op
import scanpy as sc
import numpy as np
import pandas as pd




In [3]:


def read_all_results_grnboost(root_dir, select = ('overlaps_global_top_k_quantile_count.tsv')):
    """
    Reads all YAML files in a tree of directories starting from root_dir.extended
    Args: 
        root_dir (str): The path to the root directory.

    Returns:
        list: A list of dictionaries, where each dictionary represents the
              content of a YAML file.
    """
    all_results = []
    
    # Normalize the root_dir path
    root_dir = os.path.abspath(root_dir)

    # Check if the root directory exists
    if not os.path.isdir(root_dir):
        print(f"Error: Directory '{root_dir}' not found.")
        return all_results

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith(select) :

                filepath = os.path.join(dirpath, filename)
                try:
                    overlaps = pd.read_csv(filepath, sep = '\t', index_col=0)
                    overlaps['dataset'] = op.basename(op.dirname(filepath))

                    all_results.append(overlaps)
                except:
                    continue
    all_results = pd.concat(all_results)

                    
    return all_results


def read_all_results_sweep(root_dir, select = ('overlaps_global_top_k.tsv')):
    """
    Reads all YAML files in a tree of directories starting from root_dir.extended
    Args: 
        root_dir (str): The path to the root directory.

    Returns:
        list: A list of dictionaries, where each dictionary represents the
              content of a YAML file.
    """
    all_results = []
    
    # Normalize the root_dir path
    root_dir = os.path.abspath(root_dir)

    # Check if the root directory exists
    if not os.path.isdir(root_dir):
        print(f"Error: Directory '{root_dir}' not found.")
        return all_results

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith(select) :

                filepath = os.path.join(dirpath, filename)
                try:
                    overlaps = pd.read_csv(filepath, sep = '\t', index_col=0)
                    overlaps['dataset'] = op.basename(op.dirname(filepath))
                    overlaps['method']  = op.basename(filepath).replace(select, '')
                    all_results.append(overlaps)
                except:
                    continue
    all_results = pd.concat(all_results)

                    
    return all_results


In [4]:
final_metrics = read_all_results_sweep('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_best_models_log3/',   select = ('_aggregated_performance.tsv'))
split_data = final_metrics['method'].str.split('_', expand=True)
split_data.columns = ['xai_method', 'hidden_layer_size', 'n_hidden_layers', 'dropout', 'model', 'background', 'raw']
final_metrics = pd.concat([final_metrics, split_data], axis=1)
final_metrics  = final_metrics.drop(columns= ['n_top'])
final_metrics = final_metrics.rename(columns= {'top_perc': "n_top", 'percentage_recovered': 'percentage_overlap',  'type': 'target'})
final_metrics.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/edge_recovery_best_netmap_log3.tsv', sep = '\t', index = False)

In [4]:
final_metrics_sc = read_all_results_sweep('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_scgenerai/',   select = ('_aggregated_performance.tsv'))


In [6]:

final_metrics_sc  = final_metrics_sc.drop(columns= ['n_top'])
final_metrics_sc['method'] = 'scgenerai_config'
final_metrics_sc = final_metrics_sc.rename(columns= {'top_perc': "n_top", 'percentage_recovered': 'percentage_overlap',  'type': 'target'})
final_metrics_sc.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/edge_recovery_scgenerai.tsv', sep = '\t', index = False)

In [7]:
grnb = read_all_results_grnboost('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/',   select = 'preclustered_overlaps_global_top_k.tsv')
grnb['top_perc'] = np.tile([0.001,0.01, 0.05, 0.1, 0.2, 0.25, 0.5, 0.75, 1.0], reps = int(grnb.shape[0]/9))
grnb  = grnb.drop(columns= ['n_top'])
grnb = grnb.rename(columns= {'top_perc': "n_top", 'percentage_recovered': 'percentage_overlap', 'config': 'method', 'type': 'target'})
grnb.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/edge_recovery_grnboost.tsv', sep = '\t', index=False)
